<a href="https://colab.research.google.com/github/wtree101/ZIP-RC-Colab/blob/main/notebooks/colab/08_graduation_decision.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Step 8 — 0.6B 实验毕业决策

汇总所有 stage report，给出是否值得升级到 1.7B 的明确结论。

**毕业条件：** predictor discrimination、remaining-length signal、controller 至少部分 budget 不输 fixed baseline。任何 operational gate 失败都应先修流水线。

先运行 `colab/00_memory_and_config.ipynb`；本 Notebook 的训练命令会自动使用 ZIP mamba 环境。

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO = Path("/content/ZIP-RC-Colab")
ZIP_PY = Path("/content/mamba/envs/zip/bin/python")

if not REPO.exists():
    raise FileNotFoundError("远端仓库不存在；请先运行 colab/00_memory_and_config.ipynb。")
if not ZIP_PY.exists():
    raise FileNotFoundError("ZIP Python 环境不存在；请先运行 colab/00_memory_and_config.ipynb。")

os.environ["ZIPRC_PYTHON"] = str(ZIP_PY)
sys.path.insert(0, str(REPO / "notebooks"))
from ziprc_notebook_utils import *

CONFIG = load_config(REPO)
print("Repository:", REPO)
print("ZIP Python:", ZIP_PY)
print("Experiment:", CONFIG["experiment_name"])

In [ ]:
import json
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display, Markdown

report_dir = REPO / "artifacts" / "stage_reports"
expected = [
    "00_memory_and_config", "01_pilot_generation", "02_pilot_quality",
    "03_training_data", "04_prompt_split", "05_predictor_training",
    "06_predictor_evaluation", "07_controller_comparison",
]
reports = {}
for name in expected:
    path = report_dir / f"{name}.json"
    reports[name] = json.loads(path.read_text(encoding="utf-8")) if path.exists() else None

summary = pd.DataFrame([
    {
        "stage": name,
        "report_exists": report is not None,
        "operational": report["operational_passed"] if report else False,
        "scientific": report["scientific_passed"] if report else False,
    }
    for name, report in reports.items()
])
display(summary)

ax = summary.set_index("stage")[["operational", "scientific"]].astype(int).plot.barh(figsize=(10, 6), color=["#4c78a8", "#49beaa"])
ax.set(xlim=(0, 1.05), title="ZIP-RC 0.6B stage gates", xlabel="pass (1) / review (0)")
plt.tight_layout()
plt.show()

In [ ]:
operational_ok = bool(summary["report_exists"].all() and summary["operational"].all())
predictor_ok = bool(summary.loc[summary["stage"] == "06_predictor_evaluation", "scientific"].iloc[0]) if reports["06_predictor_evaluation"] else False
controller_ok = bool(summary.loc[summary["stage"] == "07_controller_comparison", "scientific"].iloc[0]) if reports["07_controller_comparison"] else False
graduate = operational_ok and predictor_ok and controller_ok

if graduate:
    display(Markdown("## ✅ 建议升级到 Qwen3-1.7B\n0.6B 已证明 predictor 和 controller 均有可用信号。"))
elif not operational_ok:
    display(Markdown("## ⛔ 暂停扩模\n至少一个流水线阶段未妥善完成，请先处理 operational gate。"))
else:
    display(Markdown("## ⚠️ 保持 0.6B 调试\n流水线正常，但 discrimination / length / controller 信号尚未全部达到毕业条件。"))

failed_gates = []
for stage, report in reports.items():
    if report:
        failed_gates.extend({"stage": stage, **item} for item in report["gates"] if not item["passed"])
display(pd.DataFrame(failed_gates) if failed_gates else pd.DataFrame([{"status": "all gates passed"}]))

checks = [
    gate("所有阶段报告存在", summary["report_exists"].all(), f"{summary['report_exists'].sum()}/{len(summary)}"),
    gate("所有 operational gates 通过", operational_ok, "流水线完整性"),
    gate("Predictor 达标", predictor_ok, "discrimination + incorrect recall + length", kind="scientific"),
    gate("Controller 达标", controller_ok, "至少出现一个 Pareto signal", kind="scientific"),
]
display(gate_frame(checks))
save_stage_report(REPO, "08_graduation_decision", checks, {"graduate_to_1_7b": graduate})